# Example 3: Real LLM Quantization with bitsandbytes

This notebook demonstrates quantizing actual language models using bitsandbytes:
- Loading models in 8-bit and 4-bit
- Memory usage comparison
- Inference speed benchmarking
- Quality assessment

**Note:** First run downloads a model (~700MB). Requires 6GB+ GPU VRAM.

## Setup and Dependencies Check

In [ ]:
import torch
import time
import os

# Suppress warnings
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ GPU not available. This example requires CUDA.")

In [ ]:
# Check bitsandbytes
try:
    import bitsandbytes as bnb
    print(f"✓ bitsandbytes {bnb.__version__}")
except ImportError:
    print("❌ bitsandbytes not installed. Run: pip install bitsandbytes")

try:
    import transformers
    print(f"✓ transformers {transformers.__version__}")
except ImportError:
    print("❌ transformers not installed. Run: pip install transformers")

try:
    import accelerate
    print(f"✓ accelerate {accelerate.__version__}")
except ImportError:
    print("❌ accelerate not installed. Run: pip install accelerate")

## Configuration

We'll use **facebook/opt-350m** (~350M parameters, ~700MB in FP16).  
You can also try: `facebook/opt-125m` (smaller) or `facebook/opt-1.3b` (larger, needs more VRAM).

In [ ]:
MODEL_NAME = "facebook/opt-350m"
TEST_PROMPT = "The future of artificial intelligence is"

print(f"Model: {MODEL_NAME}")
print(f"Test prompt: '{TEST_PROMPT}'")
print("\n⏳ Note: First run will download the model...")

## Part 1: Load Model in FP16 (Baseline)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

def get_gpu_memory():
    """Get current GPU memory allocated in GB."""
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1e9
    return 0

def reset_memory_tracking():
    """Reset memory stats for accurate tracking."""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

# Clear memory and reset tracking
reset_memory_tracking()

# Load FP16 model
print("Loading FP16 model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Get memory after loading
mem_fp16 = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0

print(f"✓ FP16 model loaded")
print(f"✓ GPU memory used: {mem_fp16:.2f} GB")

In [ ]:
# Test generation
inputs = tokenizer(TEST_PROMPT, return_tensors="pt").to(model_fp16.device)

with torch.no_grad():
    outputs = model_fp16.generate(**inputs, max_new_tokens=50, do_sample=False)

output_fp16 = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("FP16 Output:")
print(output_fp16)

## Part 2: Load Model in INT8

In [ ]:
from transformers import BitsAndBytesConfig

# Clean up
del model_fp16
reset_memory_tracking()

# Configure INT8 quantization
bnb_config_8bit = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,  # Default threshold for outlier detection
)

print("Loading INT8 model with bitsandbytes...")

model_8bit = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config_8bit,
    device_map="auto"
)

# Get memory after loading
mem_8bit = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0

print(f"✓ INT8 model loaded")
print(f"✓ GPU memory used: {mem_8bit:.2f} GB")

if mem_fp16 > 0 and mem_8bit > 0:
    reduction = mem_fp16 / mem_8bit
    savings = ((mem_fp16 - mem_8bit) / mem_fp16) * 100
    print(f"✓ Memory reduction: {reduction:.2f}x ({savings:.1f}% savings)")
elif mem_8bit <= 0:
    print("⚠️ Memory tracking issue - showing model size from parameters instead")
    # Fallback: calculate from model parameters
    param_size = sum(p.numel() * p.element_size() for p in model_8bit.parameters()) / 1e9
    print(f"✓ Estimated size from parameters: {param_size:.2f} GB")

In [ ]:
# Test generation
inputs = tokenizer(TEST_PROMPT, return_tensors="pt").to(model_8bit.device)

with torch.no_grad():
    outputs = model_8bit.generate(**inputs, max_new_tokens=50, do_sample=False)

output_8bit = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("INT8 Output:")
print(output_8bit)

## Part 3: Load Model in NF4 (4-bit)

In [ ]:
from transformers import BitsAndBytesConfig

# Clean up
del model_8bit
reset_memory_tracking()

# Configure 4-bit quantization
bnb_config_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,  # Nested quantization for additional savings
)

print("Loading NF4 model...")

model_4bit = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config_4bit,
    device_map="auto"
)

# Get memory after loading
mem_4bit = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0

print(f"✓ NF4 model loaded")
print(f"✓ GPU memory used: {mem_4bit:.2f} GB")

if mem_fp16 > 0 and mem_4bit > 0:
    reduction = mem_fp16 / mem_4bit
    savings = ((mem_fp16 - mem_4bit) / mem_fp16) * 100
    print(f"✓ Memory reduction: {reduction:.2f}x ({savings:.1f}% savings)")
elif mem_4bit <= 0:
    print("⚠️ Memory tracking issue - showing model size from parameters instead")
    # Fallback: calculate from model parameters
    param_size = sum(p.numel() * p.element_size() for p in model_4bit.parameters()) / 1e9
    print(f"✓ Estimated size from parameters: {param_size:.2f} GB")

In [ ]:
# Test generation
inputs = tokenizer(TEST_PROMPT, return_tensors="pt").to(model_4bit.device)

with torch.no_grad():
    outputs = model_4bit.generate(**inputs, max_new_tokens=50, do_sample=False)

output_4bit = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("NF4 Output:")
print(output_4bit)

## Summary Comparison

In [ ]:
print("\n" + "="*70)
print("MEMORY USAGE SUMMARY")
print("="*70)
print(f"{'Precision':<15} {'Memory (GB)':<15} {'Savings':<15}")
print("-"*70)

if mem_fp16:
    print(f"{'FP16':<15} {mem_fp16:<15.2f} {'Baseline':<15}")
if mem_8bit:
    savings_8 = f"{((mem_fp16-mem_8bit)/mem_fp16*100):.1f}%" if mem_fp16 else "-"
    print(f"{'INT8':<15} {mem_8bit:<15.2f} {savings_8:<15}")
if mem_4bit:
    savings_4 = f"{((mem_fp16-mem_4bit)/mem_fp16*100):.1f}%" if mem_fp16 else "-"
    print(f"{'NF4':<15} {mem_4bit:<15.2f} {savings_4:<15}")

print("\n✓ All outputs should be very similar, showing minimal quality loss!")

## Key Takeaways

1. ✅ **INT8** reduces memory by ~2x with minimal quality loss
2. ✅ **NF4** reduces memory by ~4x, still maintains good quality
3. ✅ **bitsandbytes** handles all complexity automatically
4. ✅ Output quality remains very close across all precisions
5. ✅ Enables running larger models on consumer GPUs

### Next Steps:
- Try with larger models (opt-1.3b, opt-2.7b)
- Experiment with different quantization configs
- Run notebook 04 for GPTQ quantization
- Try fine-tuning with QLoRA (Parameter Efficient Fine-Tuning)